In [2]:
import pandas as pd 
df=pd.read_csv("monitor_data.csv")
df.head()


,Time,Voltage(V),Current(A),Power(W),Leakage Current(mA),total_forward_energy(kWh),price(DZD)
0,2025-08-12 22:30:01,355.92,7.68,24.2,0,16.6,88.5372
1,2025-08-12 22:30:11,343.12,7.68,22.2,0,16.6,88.5372
2,2025-08-12 22:30:21,358.48,7.68,21.2,0,16.6,88.5372
3,2025-08-12 22:30:31,373.84,7.68,20.2,0,16.6,88.5372
4,2025-08-12 22:30:41,373.84,7.68,20.2,0,16.6,88.5372


In [1]:
import os
import time
import base64
import numpy as np
import sounddevice as sd
import soundfile as sf
import whisper
import pyttsx3
from tuya_connector import TuyaOpenAPI


# ---------------- CONFIG ----------------
ACCESS_ID = os.getenv("TUYA_ACCESS_ID", "8dued99qjs7sryafpa8c")
ACCESS_KEY = os.getenv("TUYA_ACCESS_KEY", "173154a0470d4661802c6aafb94a178f")
API_ENDPOINT = os.getenv("TUYA_API_ENDPOINT", "https://openapi.tuyaeu.com")
DEVICE_ID = os.getenv("TUYA_DEVICE_ID", "bf0178074ad9c548aduc79")

MODEL_NAME = os.getenv("WHISPER_MODEL", "base")   # whisper model size (tiny, base, small, medium, large)
AUDIO_FILE = os.getenv("AUDIO_FILE", "command.wav")
SAMPLE_RATE = 16000  # keep 16 kHz so we can avoid ffmpeg

# ---------------- INIT ----------------
# Init Tuya Cloud
openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)

# Init TTS
tts = pyttsx3.init()

def speak(text: str):
    print("[AGENT]:", text)
    try:
        tts.say(text)
        tts.runAndWait()
    except Exception as e:
        print("[TTS ERROR]:", e)


def try_connect_tuya():
    try:
        openapi.connect()
        return True
    except Exception as e:
        speak(f"Tuya connection failed: {e}")
        return False

# ============= TUYA FUNCTIONS =============
def get_measurements():
    """Fetch voltage/current/power/energy/leakage.
    Returns tuple (voltage, current, power, energy, leakage) or (None,...)
    """
    try:
        resp = openapi.get(f"/v1.0/devices/{DEVICE_ID}/status")
        voltage = current = power = energy = leakage = None
        for item in (resp or {}).get("result", []) :
            code = item.get("code")
            val = item.get("value")
            if code == "phase_a" and isinstance(val, str):
                # Device reports phase A metrics packed as base64 bytes
                raw = base64.b64decode(val)
                # Basic index mapping (may vary by device model)
                try:
                    voltage = (raw[2] | (raw[3] << 8)) / 10.0
                    current = (raw[4] | (raw[5] << 8)) / 100.0
                    power   = (raw[6] | (raw[7] << 8)) / 10.0
                except Exception:
                    pass
            elif code == "total_forward_energy":
                try:
                    energy = float(val) / 100.0
                except Exception:
                    energy = None
            elif code == "leakage_current":
                leakage = val
        return voltage, current, power, energy, leakage
    except Exception as e:
        speak(f"Failed to read measurements: {e}")
        return None, None, None, None, None


def set_switch(state: bool):
    try:
        commands = {'commands': [{'code': 'switch', 'value': state}]}
        openapi.post(f"/v1.0/devices/{DEVICE_ID}/commands", commands)
        speak(f"Breaker turned {'ON' if state else 'OFF'}.")
    except Exception as e:
        speak(f"Failed to set switch: {e}")

# ============= SPEECH FUNCTIONS =============
def record_audio(filename: str, duration: int = 4, samplerate: int = SAMPLE_RATE):
    """Record voice from mic into a WAV file."""
    speak("Listening...")
    try:
        recording = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype="float32")
        sd.wait()
        # soundfile expects float32 in range [-1, 1]
        sf.write(filename, recording, samplerate)
        return filename
    except Exception as e:
        speak(f"Recording failed: {e}")
        return None


def transcribe_audio(filename: str) -> str:
    """Transcribe voice with Whisper WITHOUT ffmpeg.
    We read audio with soundfile and pass raw samples directly to Whisper.
    """
    if not filename or not os.path.exists(filename):
        speak("Audio file not found.")
        return ""

    try:
        # Load the model once and reuse (global cache)
        global _whisper_model
        if '_whisper_model' not in globals() or _whisper_model is None:
            _whisper_model = whisper.load_model(MODEL_NAME)

        data, sr = sf.read(filename, dtype="float32")  # returns float32, mono/mono-like
        if data.ndim > 1:
            data = data[:, 0]  # take first channel just in case

        # Ensure 16kHz for Whisper when bypassing ffmpeg
        if sr != SAMPLE_RATE:
            # simple resample using linear interpolation (good enough for commands)
            target_len = int(len(data) * SAMPLE_RATE / sr)
            data = np.interp(
                np.linspace(0, len(data), target_len, endpoint=False),
                np.arange(len(data)),
                data,
            ).astype(np.float32)

        result = _whisper_model.transcribe(data)
        return (result.get("text") or "").strip().lower()

    except Exception as e:
        speak(f"Transcription failed: {e}")
        return ""

# ============= HANDLER =============

def handle_query(text: str):
    t = (text or "").lower()
    if any(phrase in t for phrase in ["turn on", "power on", "allume" , "on"]):
        set_switch(True)
    elif any(phrase in t for phrase in ["turn off", "power off", "éteins", "eteins" , "off"]):
        set_switch(False)
    elif any(word in t for word in ["status", "data", "measurements", "voltage", "current", "power", "energy", "leakage"]):
        v, i, p, e, l = get_measurements()
        if v is None and i is None and p is None and e is None and l is None:
            speak("Couldn't fetch measurements.")
        else:
            speak(f"Voltage {v} volts, Current {i} amps, Power {p} watts, Energy {e} kilowatt hours, Leakage {l} milliamps.")
    elif any(kw in t for kw in ["quit", "exit", "stop agent"]):
        speak("Goodbye!")
        raise SystemExit
    else:
        speak("Sorry, I did not understand.")


if __name__ == "__main__":
    speak("Voice agent is starting...")

    if not all([ACCESS_ID and ACCESS_ID != "3rtkjvnvn7echqjc4ffn",
                ACCESS_KEY and ACCESS_KEY != "93ad1b54d5544c64b5e893de55afab4a",
                DEVICE_ID and DEVICE_ID != "bf0178074ad9c548aduc79"]):
        speak("WARNING: Tuya credentials are not set. Set environment variables before using Tuya features.")

    # Connect to Tuya (won't crash if it fails)
    try_connect_tuya()

    speak("Voice agent is ready.")

    while True:
        try:
            input("Press Enter to give a command...")
            audio_path = record_audio(AUDIO_FILE)
            if not audio_path:
                
                continue
            text = transcribe_audio(audio_path)
            print("audio:", audio_path)
            print("[YOU]:", text)
            if text:
                handle_query(text)
        except KeyboardInterrupt:
            speak("Interrupted by user. Bye!")
            break
        except SystemExit:
            break
        except Exception as e:
            speak(f"Unexpected error: {e}")
            time.sleep(5)


[AGENT]: Voice agent is starting...
[AGENT]: WARNING: Tuya credentials are not set. Set environment variables before using Tuya features.
[AGENT]: Voice agent is ready.
[AGENT]: Listening...


c:\Users\dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


audio: command.wav
[YOU]: 
[AGENT]: Listening...
audio: command.wav
[YOU]: for the
[AGENT]: Sorry, I did not understand.
[AGENT]: Listening...
audio: command.wav
[YOU]: voltège ... zelke!
[AGENT]: Sorry, I did not understand.
[AGENT]: Listening...
audio: command.wav
[YOU]: measurements
[AGENT]: Voltage 153.6 volts, Current 2.54 amps, Power 1331.3 watts, Energy 977.39 kilowatt hours, Leakage 0 milliamps.
[AGENT]: Listening...
audio: command.wav
[YOU]: of agent.
[AGENT]: Sorry, I did not understand.
[AGENT]: Listening...
audio: command.wav
[YOU]: stop agent!
[AGENT]: Goodbye!
